# BiLSTM phân loại cảm xúc UIT-VSFC

UIT-VSFC là 16 175 câu phản hồi của sinh viên về giảng viên, chương trình, cơ sở vật chất, gán 3 nhãn cảm xúc
(0 negative, 1 neutral, 2 positive). Bạn dựng pipeline phân loại hoàn chỉnh bằng PyTorch: vocab, padding, **packing**,
LSTM hai chiều, train có chọn epoch theo validation, và so với hai baseline rẻ. Bài Basic 03 đã tự viết LSTM bằng numpy;
ở đây dùng `nn.LSTM` (cùng phương trình, chạy nhanh hơn nhiều).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
- Chấm: `uv run pytest tracks/ai-engineer/modules/04-nlp-sequence-models/exercises/intermediate/02-lstm-sentiment -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải UIT-VSFC nếu chưa có (khoảng 1 MB)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy/torch
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
print(sorted(p.relative_to(VSFC).as_posix() for p in VSFC.rglob("*.parquet")))

In [ ]:
import copy
import time
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD, UNK = 0, 1
LABELS = ["negative", "neutral", "positive"]

## 1. Dữ liệu, vocab, batch có padding

- `load_split(root, split) -> (sentences, labels)`: đọc `root / "default" / split / "0000.parquet"` (`split` là `train`,
  `validation` hoặc `test`), trả danh sách câu (cột `sentence`) và nhãn `np.int64` (cột `sentiment`).
- `tokenize(sentence) -> list[str]`: NFC, `lower()`, tách theo khoảng trắng (dữ liệu đã tách dấu câu sẵn).
- `build_vocab(sentences, min_freq) -> dict[str, int]`: `<pad>` là 0, `<unk>` là 1, sau đó các âm tiết xuất hiện
  `>= min_freq` lần, theo tần suất giảm dần, hoà thì theo thứ tự chữ cái.
- `encode_batch(sentences, vocab, max_len) -> (ids, lengths)`: `ids` là `LongTensor (N, L)` với `L` là độ dài lớn
  nhất **trong batch** (sau khi cắt ở `max_len`), pad bằng 0 ở cuối; âm tiết lạ thành `<unk>`; câu rỗng thành một token
  `<unk>`. `lengths` là `LongTensor (N,)` độ dài thật.

In [ ]:
def load_split(root, split: str) -> tuple[list[str], np.ndarray]:
    # TODO
    raise NotImplementedError


def tokenize(sentence: str) -> list[str]:
    # TODO
    raise NotImplementedError


def build_vocab(sentences: list[str], min_freq: int) -> dict[str, int]:
    # TODO: <pad> = 0, <unk> = 1, rồi theo tần suất giảm dần, hoà theo chữ cái
    raise NotImplementedError


def encode_batch(sentences: list[str], vocab: dict[str, int], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: phân bố nhãn, độ dài câu, vocab
train_s, train_y = load_split(VSFC, "train")
val_s, val_y = load_split(VSFC, "validation")
test_s, test_y = load_split(VSFC, "test")
for name, y in [("train", train_y), ("validation", val_y), ("test", test_y)]:
    print(f"{name:10} {len(y):6} câu, nhãn {np.bincount(y).tolist()}")
lens = np.array([len(tokenize(s)) for s in train_s])
print("âm tiết mỗi câu: trung vị", np.median(lens), "| p99", np.percentile(lens, 99), "| max", lens.max())
vocab = build_vocab(train_s, min_freq=2)
print("vocab (kể cả <pad>, <unk>):", len(vocab))
for label in range(3):
    print(LABELS[label], "→", [s for s, y in zip(train_s[:200], train_y[:200]) if y == label][:3])

## 2. LSTM hai chiều có packing

`LSTMClassifier(vocab_size, emb_dim, hidden, n_classes, bidirectional=True, dropout=0.3)`:

- `nn.Embedding(vocab_size, emb_dim, padding_idx=0)` → dropout → `nn.LSTM(emb_dim, hidden, batch_first=True,
  bidirectional=bidirectional)` → nối hidden state **cuối** của mỗi chiều (chiều xuôi: sau token thật cuối cùng; chiều
  ngược: sau khi đọc về token đầu) → dropout → `nn.Linear` ra `n_classes` logits.
- `forward(ids, lengths)`: dùng `nn.utils.rnn.pack_padded_sequence(..., lengths, batch_first=True,
  enforce_sorted=False)`. Kết quả của một câu **không được phụ thuộc** câu đó nằm chung batch với câu dài bao nhiêu (test
  kiểm điều này). `h_n` của `nn.LSTM` có shape `(num_directions, N, hidden)`.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, n_classes: int, bidirectional: bool = True,
                 dropout: float = 0.3):
        super().__init__()
        # TODO: embedding (padding_idx=0), dropout, LSTM batch_first, Linear
        raise NotImplementedError

    def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # TODO: pack_padded_sequence, nối h_n của các chiều
        raise NotImplementedError

## 3. Metric, train có chọn epoch, dự đoán

- `macro_f1(y_true, y_pred, n_classes) -> float`: trung bình F1 của từng lớp, tự viết bằng numpy (không dùng
  sklearn). Lớp không có trong cả nhãn lẫn dự đoán thì F1 tính là 0, giống `sklearn` với `zero_division=0`.
- `predict(model, ids, lengths, batch_size=512) -> np.ndarray`: ở chế độ `eval`, không tính gradient, trả nhãn argmax.
- `train_classifier(model, train, val, epochs, lr, batch_size, seed, class_weight=None) -> dict`: `train`, `val` là bộ
  `(ids, lengths, labels)` (labels là `LongTensor`).
  - `torch.manual_seed(seed)` ở đầu; xáo mỗi epoch bằng `torch.randperm(N, generator=g)` với
    `g = torch.Generator().manual_seed(seed)`.
  - Adam `lr`, cross-entropy (có `weight=class_weight` nếu truyền), clip gradient chuẩn 1.0.
  - Cuối mỗi epoch tính macro-F1 trên `val`; giữ **bản sao** trọng số của epoch tốt nhất và nạp lại trước khi trả.
  - Trả `{"train_loss": [...], "val_f1": [...]}`, mỗi phần tử một epoch, train loss trung bình theo mẫu.

In [ ]:
def macro_f1(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    # TODO
    raise NotImplementedError


@torch.no_grad()
def predict(model: nn.Module, ids: torch.Tensor, lengths: torch.Tensor, batch_size: int = 512) -> np.ndarray:
    # TODO
    raise NotImplementedError


def train_classifier(model: nn.Module, train: tuple, val: tuple, epochs: int, lr: float, batch_size: int, seed: int,
                     class_weight: torch.Tensor | None = None) -> dict[str, list[float]]:
    # TODO: seed, randperm với generator, Adam, clip 1.0, chọn epoch theo val macro-F1, nạp lại trọng số tốt nhất
    raise NotImplementedError

In [ ]:
# explore: hai baseline rẻ: đoán lớp nhiều nhất, TF-IDF 1-2 gram + logistic regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

results = {}
majority = np.full(len(test_y), np.bincount(train_y).argmax())
results["đoán lớp nhiều nhất"] = majority
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = LogisticRegression(C=10, max_iter=2000).fit(vec.fit_transform(train_s), train_y)
results["TF-IDF + LR"] = clf.predict(vec.transform(test_s))

In [ ]:
# explore: LSTM một chiều và BiLSTM, 6 epoch, chọn epoch theo val macro-F1 (khoảng 1-2 phút mỗi model trên CPU)
torch.set_num_threads(4)
train_t = (*encode_batch(train_s, vocab, 64), torch.tensor(train_y))
val_t = (*encode_batch(val_s, vocab, 64), torch.tensor(val_y))
test_ids, test_len = encode_batch(test_s, vocab, 64)
for name, bi in [("LSTM một chiều", False), ("BiLSTM", True)]:
    torch.manual_seed(0)
    model = LSTMClassifier(len(vocab), 128, 128, 3, bidirectional=bi)
    start = time.perf_counter()
    hist = train_classifier(model, train_t, val_t, epochs=6, lr=2e-3, batch_size=64, seed=0)
    best = int(np.argmax(hist["val_f1"]))
    print(f"{name}: val macro-F1 theo epoch {[round(f, 4) for f in hist['val_f1']]}, chọn epoch {best + 1},"
          f" {time.perf_counter() - start:.0f} s")
    results[name] = predict(model, test_ids, test_len)

In [ ]:
# explore: bảng kết quả trên test
print(f"{'model':22} {'acc':>6} {'macro-F1':>9} {'F1 neg':>7} {'F1 neu':>7} {'F1 pos':>7}")
for name, pred in results.items():
    per = f1_score(test_y, pred, average=None, labels=[0, 1, 2], zero_division=0)
    print(f"{name:22} {np.mean(pred == test_y):6.4f} {macro_f1(test_y, pred, 3):9.4f}"
          f" {per[0]:7.3f} {per[1]:7.3f} {per[2]:7.3f}")
print(confusion_matrix(test_y, results["BiLSTM"]))

In [ ]:
# explore: phân tích lỗi: câu neutral bị đoán sai
pred = results["BiLSTM"]
wrong = [(s, LABELS[p]) for s, y, p in zip(test_s, test_y, pred) if y == 1 and p != 1]
print(f"{len(wrong)} / {np.sum(test_y == 1)} câu neutral bị đoán sai, ví dụ:")
for s, p in wrong[:10]:
    print(f"  [{p}] {s}")

## 4. Đọc kết quả

So với bảng ở mục 5 phần lý thuyết. Accuracy của mọi model (trừ baseline đoán bừa) gần như bằng nhau; khác biệt nằm ở
lớp neutral, chỉ 5% dữ liệu và nhiều câu neutral thật ra mơ hồ (đọc vài câu bị đoán sai ở trên). Vì vậy macro-F1, không
phải accuracy, là metric để chọn model. Muốn cải thiện neutral, thử `class_weight` (tham số của `train_classifier`), gộp
dữ liệu, hoặc mô hình tiền huấn luyện (PhoBERT, LLM ở module 11) và đo lại bằng đúng metric này.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.